# 构建模型并行
- 在 Megatron 的 3D 并行（张量并行 TP + 流水线并行 PP + 数据并行 DP）中，<font color='red'>tp-pp' 代表模型并行（Model Parallelism）</font>
  - 它的作用是将所有的 GPU 卡（World）划分成若干个组。<font color='red'>每个组内的 GPU 共同协作，负责训练同一个完整模型的副本。</font>
  - 组内的卡通过 TP（切分单层模型参数）和 PP（切分模型层数）来容纳一个巨大的模型。
  
  
💡 举个例子方便理解
假设你一共有 16 张 GPU 卡（编号 g0 到 g15），设置了 张量并行度 TP=2，流水线并行度 PP=4。
那么，数据并行度 DP = 16 / (2 * 4) = 2。

- 这段代码执行时，decoder_rank_generator.get_ranks('tp-pp') 会生成 2 个模型并行组（因为 DP=2，相当于有两个完整的模型副本在并行训练）：
- 组 1 (ranks)：[g0, g1, g4, g5, g8, g9, g12, g13]
- 组 2 (ranks)：[g2, g3, g6, g7, g10, g11, g14, g15]

<font color='red'>如果当前运行代码的是 g0 号卡：</font>
1. 循环第一次拿到 [g0, g1, ...]。
2. 创建这个组的通信对象 group。
3. 发现 g0 在这个列表里，于是 g0 认领这个组，将 _MODEL_PARALLEL_GROUP 指向它。
4. 循环第二次拿到 [g2, g3, ...]，g0 不在里面，跳过。

最终，g0 知道自己属于第一个模型并行组，后续它只需要和组内的 g1, g4, g5 等卡进行模型内部的通信，而与 g2, g3 等卡完全隔离（它们属于另一个数据并行组）。- 

In [ ]:
    '''
    构建模型并行组：
    
    '''
    # Build the model-parallel groups.
    global _MODEL_PARALLEL_GROUP
    global _MODEL_PARALLEL_GLOBAL_RANKS
    assert _MODEL_PARALLEL_GROUP is None, 'model parallel group is already initialized'
    '''
    decoder_rank_generator 会遍历生成所有的模型并行组，但当前这张卡只需要知道自己属于哪一个组。
    '''
    for ranks in decoder_rank_generator.get_ranks('tp-pp'):
        
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("mp", nccl_comm_cfgs), #  指定了使用 NCCL 后端进行高性能的 GPU 间通信。
            group_desc="MODEL_PARALLEL_GROUP",
        )
        
        '''
        rank：代表当前进程（当前这张 GPU 卡）的全局编号。
        
        当循环到包含当前卡号的那一组时，条件成立，当前卡就会把该组的通信句柄（group）和组内所有卡的编号（ranks）保存到全局变量中。
        '''
        if rank in ranks:
            _MODEL_PARALLEL_GROUP = group
            _MODEL_PARALLEL_GLOBAL_RANKS = ranks


## decoder_rank_generator

In [ ]:
    '''
    Rank 生成器
    
    这是构建并行环境的“大脑”
    
    在分布式训练中，每张卡有一个物理 ID（Rank）。
    RankGenerator 负责告诉程序：“哪些物理 ID 属于同一个 TP 组”，“哪些属于同一个 DP 组”
    
    tp: 张量并行
    pp: 流水线并行
    dp: 数据并行
    cp: 上下文并行（用于超长文本）
    ep: 专家并行（MoE 模型）

    Order 参数：order="tp-cp-ep-dp-pp" 决定了多维并行下的映射顺序。
               不同的顺序（如 PP 在前还是 DP 在前）会直接影响通信的拓扑结构和效率。
    '''
    decoder_rank_generator = RankGenerator(
        tp=tensor_model_parallel_size,
        ep=1,
        dp=data_parallel_size,
        pp=pipeline_model_parallel_size,
        cp=context_parallel_size,
        order=order,
        rank_offset=rank_offset,
    )